In [1]:
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from PIL import Image
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

from torch.utils.data import Dataset, DataLoader


np.random.seed(42)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DataLoader

In [2]:
dataset = load_dataset("andreribeiro87/kvasir-seg-augmented")

In [3]:
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

In [4]:
train_data

Dataset({
    features: ['image', 'mask'],
    num_rows: 4800
})

In [5]:
def mask_to_points(masks, batch_size, pos_count=3, neg_count=3):
    return_points = []
    return_labels = []
    for i in range(batch_size):
        mask = masks[i]
        ys, xs = np.where(mask > 0)
        pos_points = []
        for i in range(pos_count):
            rand_ind = np.random.randint(len(ys))
            pos_points.append([xs[rand_ind], ys[rand_ind]])
        neg_points = []
        ys, xs = np.where(mask == 0)
        for i in range(neg_count):
            rand_ind = np.random.randint(len(ys))
            neg_points.append([xs[rand_ind], ys[rand_ind]])
        labels = np.concat([np.ones(pos_count), np.zeros(neg_count)])
        return_points.append(np.concat([pos_points, neg_points]))
        return_labels.append(labels)
    return return_points, return_labels

In [6]:
def read_sample(data):
    entry = data[np.random.randint(len(data))]

    img = np.array(entry['image'].convert("RGB"))
    mask = np.array(entry['mask'].convert("L"))
    mask = (mask > 127).astype(np.uint8)

    return img, mask

In [7]:
def read_batch(data, batch_size=4, points_count=3):
    images = []
    masks = []
    points = []
    labels = []
    for i in range(batch_size):
        image, mask = read_sample(data)
        images.append(image)
        masks.append(mask)
        point, label = mask_to_points(mask, points_count, points_count)
        points.append(point)
        labels.append(label)
    return images, np.array(masks), np.array(points), np.array(labels)

In [8]:
class KvasirDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        entry = self.data[idx]
        img = np.array(entry['image'].convert("RGB"))
        mask = np.array(entry['mask'].convert("L"))
        mask = (mask > 127).astype(np.uint8)
        return img, mask

In [9]:
train_data = KvasirDataset(dataset['train'])
val_data = KvasirDataset(dataset['validation'])
test_data = KvasirDataset(dataset['test'])

In [10]:
batch_size = 4
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

# Лоссы

In [11]:
class SoftIouLoss(torch.nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps
    def forward(self, pred, mask):
        pred = torch.sigmoid(pred)
        intersection = (pred * mask).sum()
        union = pred.sum() + mask.sum() - intersection
        iou = intersection/(union + self.eps)
        return 1 - iou.mean()

In [12]:
def binary_iou_score(pred_logits, gt_mask, threshold=0.5, eps=1e-6):
    pred_prob = torch.sigmoid(pred_logits)
    pred_bin = pred_prob > threshold
    gt_bin = gt_mask > 0.5

    intersection = (pred_bin & gt_bin).sum(dim=(1, 2))
    union = (pred_bin | gt_bin).sum(dim=(1, 2))

    iou = intersection.float() / (union.float() + eps)

    return iou.mean()

In [13]:
bce_loss = torch.nn.BCEWithLogitsLoss()
iou_loss = SoftIouLoss()

In [14]:
def validate_sam2(predictor, val_data, bce_loss, iou_loss, n_samples=100):
    predictor.model.eval()

    total_loss = 0.0
    total_bce = 0.0
    total_iou_loss = 0.0
    total_iou_score = 0.0

    with torch.no_grad():
        for idx in range(n_samples):
            img, mask = val_data[idx]
            points, labels = mask_to_points(np.expand_dims(mask, axis=0), batch_size=1)

            with torch.cuda.amp.autocast(dtype=torch.float16):
                
                predictor.set_image(img)
                
                mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(
                    points, 
                    labels, 
                    box=None, 
                    mask_logits=None, 
                    normalize_coords=True
                )

        
                sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(
                    points=(unnorm_coords, labels),
                    boxes=None,
                    masks=None,
                )
        
                high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
                low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
                    image_embeddings=predictor._features["image_embed"],
                    image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=True,
                    repeat_image=False,
                    high_res_features=high_res_features
                )
                prd_masks = predictor._transforms.postprocess_masks(
                    low_res_masks, 
                    predictor._orig_hw[-1]
                )
        
                pred_logits = prd_masks[:, 0]
        
                gt_mask = torch.tensor(
                    mask[None, :, :],
                    dtype=torch.float32,
                    device=pred_logits.device,
                )
        
                loss_bce = bce_loss(pred_logits, gt_mask)
                loss_iou = iou_loss(pred_logits, gt_mask)
                loss = loss_bce + loss_iou
                iou_score = binary_iou_score(pred_logits, gt_mask)
                
            total_loss += float(loss.detach().cpu())
            total_bce += float(loss_bce.detach().cpu())
            total_iou_loss += float(loss_iou.detach().cpu())
            total_iou_score += float(iou_score.detach().cpu())
    predictor.model.train()

    return {
        "loss": total_loss / n_samples,
        "bce": total_bce / n_samples,
        "iou_loss": total_iou_loss / n_samples,
        "iou_score": total_iou_score / n_samples,
    }

# Модель

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [16]:
sam2_checkpoint = "segment-anything-2/checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)

In [17]:
for p in sam2_model.parameters():
    p.requires_grad = False

for p in sam2_model.sam_mask_decoder.parameters():
    p.requires_grad = True

In [18]:
optimizer = torch.optim.AdamW(
    params=predictor.model.parameters(),
    lr=1e-5,
    weight_decay=4e-5
)

scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_2193/1970860592.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [19]:
trainable = sum(p.numel() for p in predictor.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in predictor.model.parameters())

print("trainable:", trainable)
print("total:", total)
print("percent:", 100 * trainable / total)

trainable: 4215109
total: 38962498
percent: 10.818374632961161


# Цикл обучения

In [20]:
best_val_loss = float('inf')
num_epochs = 5

In [ ]:
for epoch in range(num_epochs):
    num_batches = len(train_loader)
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for batch, (imgs, masks) in enumerate(pbar):
        imgs = list(imgs.numpy())
        masks = masks.numpy()
        input_points, input_labels = mask_to_points(masks, batch_size=batch_size)
        optimizer.zero_grad()
    
        with torch.cuda.amp.autocast(dtype=torch.float16):
            with torch.no_grad():
                predictor.set_image_batch(imgs)
    
            mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(
                input_points, 
                input_labels, 
                box=None, 
                mask_logits=None, 
                normalize_coords=True
            )
    
            sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(
                points=(unnorm_coords, labels),
                boxes=None,
                masks=None
            )
    
            high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
            
            low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
                image_embeddings=predictor._features["image_embed"],
                image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
                sparse_prompt_embeddings=sparse_embeddings,
                dense_prompt_embeddings=dense_embeddings,
                multimask_output=True,
                repeat_image=False,
                high_res_features=high_res_features
            )
            
            prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])
            pred_logits = prd_masks[:, 0]
    
            gt_masks = torch.tensor(
                masks.astype(np.float32)
            ).cuda()
    
            loss_bce = bce_loss(pred_logits, gt_masks)
            loss_iou = iou_loss(pred_logits, gt_masks)
            loss = loss_bce + loss_iou
    
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    
    metrics = validate_sam2(
        predictor=predictor,
        val_data=val_data,
        bce_loss=bce_loss,
        iou_loss=iou_loss
    )

    print(
        "epoch:", epoch,
        "train_loss:", float(loss.detach().cpu()),
        "val_loss:", metrics["loss"],
        "val_iou:", metrics["iou_score"],
    )

    val_loss = metrics["loss"]
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model_state_dict": predictor.model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
        }, "best_sam2_medical.pth")

        print("Saved best model")

Epoch 0:   0%|                                                                                                                                                                                                      | 0/1200 [00:00<?, ?it/s]/tmp/ipykernel_2193/715977851.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16):
/usr/local/lib/python3.10/dist-packages/sam2/sam2_image_predictor.py:314: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  point_coords = torch.as_tensor(
/tmp/ipykernel_2193/3202473192.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.                  

epoch: 0 train_loss: 0.39079946279525757 val_loss: 0.31028942093253137 val_iou: 0.831945376843214
Saved best model


Epoch 1:  92%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌              | 1107/1200 [02:35<00:12,  7.57it/s]

In [ ]:
checkpoint = torch.load("best_sam2_medical.pth", map_location=device)

predictor.model.load_state_dict(checkpoint["model_state_dict"])
predictor.model.eval()

In [ ]:
validate_sam2(
    predictor=predictor,
    val_data=test_data,
    bce_loss=bce_loss,
    iou_loss=iou_loss
)